# 霸王龙（Open T-Rex Mini）行走策略训练 · Colab 版 v1.1

基于 `apirrone/Open_Duck_Playground`，任务 `flat_terrain_trex`（模型由本笔记本现场生成并注册）。

**使用步骤**：运行时 → 更改运行时类型 → GPU（T4 即可）→ 全部运行。

- v1.1 新增：训练前 **2 分钟自检**（Cell 4），失败会立即停在这一格并打印原因，不会白跑
- 训练日志实时写入 `train.log`；打包前自动检查 ONNX 产物，缺产物会打印日志尾部并中止
- 试训 1 千万步（T4 约 20–40 分钟）；确认 reward 上升后把 Cell 5 的 NUM_STEPS 改为 150_000_000 正式训练（2–4 小时）

In [ ]:
# 1. 环境：安装 uv 并克隆训练场
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
!git clone --depth 1 https://github.com/apirrone/Open_Duck_Playground
%cd Open_Duck_Playground

In [ ]:
# 2. 生成霸王龙模型（与本地 trex_sim/make_trex_model.py 同一逻辑）
import pathlib

XMLS = pathlib.Path('playground/open_duck_mini_v2/xmls')
robot = (XMLS / 'open_duck_mini_v2.xml').read_text()

# 头部增重（温和版 v2 参数；重训时可放开到 0.55）
old_head = '<inertial pos="0.00815416 -0.00390754 0.0227726" mass="0.406607"'
new_head = '<inertial pos="0.00815416 -0.00390754 0.0227726" mass="0.55"'
assert robot.count(old_head) == 1
robot = robot.replace(old_head, new_head)

# 躯干增重 + 质心后移（分段尾 + 电池后置）
old_trunk = '<inertial pos="-0.0483259 -9.97823e-05 0.0384971" mass="0.698526"'
new_trunk = '<inertial pos="-0.078 -0.0001 0.0385" mass="0.85"'
assert robot.count(old_trunk) == 1
robot = robot.replace(old_trunk, new_trunk)

# 大脚掌
assert robot.count('mass="0.07524"') == 2
robot = robot.replace('mass="0.07524"', 'mass="0.095"')

# 视觉尾巴（无碰撞，仅用于视频确认姿态）
anchor = 'mesh="body_back"'
idx = robot.index(anchor)
line_end = robot.index('/>', idx) + 2
tail = '''
        <geom name="tail_visual" type="capsule" fromto="-0.10 0 0.10 -0.34 0 0.16" size="0.032"
              rgba="0.92 0.92 0.90 1" contype="0" conaffinity="0" group="1"/>
        <geom name="tail_tip" type="capsule" fromto="-0.34 0 0.16 -0.48 0 0.20" size="0.020"
              rgba="0.92 0.92 0.90 1" contype="0" conaffinity="0" group="1"/>'''
robot = robot[:line_end] + tail + robot[line_end:]
(XMLS / 'open_trex_mini_v2.xml').write_text(robot)

scene = (XMLS / 'scene_flat_terrain.xml').read_text()
scene = scene.replace('include file="open_duck_mini_v2.xml"', 'include file="open_trex_mini_v2.xml"')
(XMLS / 'scene_trex.xml').write_text(scene)

# 3. 注册任务 flat_terrain_trex
const = pathlib.Path('playground/open_duck_mini_v2/constants.py')
text = const.read_text()
anchor = '"rough_terrain_backlash": ROUGH_TERRAIN_BACKLASH_XML,'
assert anchor in text and 'flat_terrain_trex' not in text
text = text.replace(anchor, anchor + '\n        "flat_terrain_trex": ROOT_PATH / "xmls" / "scene_trex.xml",')
const.write_text(text)
print('霸王龙模型已生成并注册任务 flat_terrain_trex')

In [ ]:
# 3. 安装依赖（GPU 版 jax 已在 pyproject 中声明为 jax[cuda12]）
!uv sync
# 锁定与本地验证一致的 mujoco_playground 版本（新版删了 _src.collision 模块，会导致训练崩溃）
# 注意：Colab 的 VIRTUAL_ENV=/usr，uv pip 默认装进系统 Python；必须显式打到项目 venv
!uv pip install --python .venv/bin/python playground==0.0.3
!.venv/bin/python -c "import jax; print(jax.devices())"

In [ ]:
# 4. 训练前自检（约 2 分钟）：在子进程里跑，退出即释放显存，不与训练抢 GPU
import pathlib

sanity = """
import jax
import numpy as np
from playground.open_duck_mini_v2 import joystick

env = joystick.Joystick(task='flat_terrain_trex')
print('observation_size:', env.observation_size, 'action_size:', env.action_size)
reset = jax.jit(env.reset)
step = jax.jit(env.step)
state = reset(jax.random.PRNGKey(0))
for i in range(20):
    state = step(state, np.zeros(env.action_size))
print(f'自检通过 reward={float(state.reward):.4f} done={bool(state.done)}')
"""
pathlib.Path('sanity_trex.py').write_text(sanity)
!.venv/bin/python sanity_trex.py && echo "可以开训了 👉 跑下一格"

In [ ]:
# 5. 开训！日志实时显示并写入 train.log（用项目 .venv 的 python，内含锁定版依赖）
NUM_STEPS = 10_000_000  # 试训；正式训练改为 150_000_000
!.venv/bin/python playground/open_duck_mini_v2/runner.py --env joystick --task flat_terrain_trex \
    --num_timesteps {NUM_STEPS} --output_dir checkpoints_trex 2>&1 | tee train.log

In [ ]:
# 6. 打包产物（checkpoint + 自动导出的 ONNX）。没有产物就打印日志尾部并中止，绝不产空包
import glob
import pathlib

onnx_files = sorted(glob.glob("checkpoints_trex/**/*.onnx", recursive=True))
if not onnx_files:
    print("❌ 没有找到 ONNX 产物，训练未完成。train.log 尾部如下：")
    log = pathlib.Path("train.log")
    print("\n".join(log.read_text(errors="ignore").splitlines()[-60:]) if log.exists()
          else "(train.log 也不存在——训练一格可能根本没跑起来)")
    raise SystemExit("训练失败，请把上面的日志截图发给 Kimi 排查")

print("找到产物：")
for f in onnx_files:
    print(" ", f, f"{pathlib.Path(f).stat().st_size/1e6:.1f} MB")
!tar czf trex_policy.tar.gz checkpoints_trex train.log
from google.colab import files
files.download("trex_policy.tar.gz")
print("✅ 已触发下载 trex_policy.tar.gz（浏览器如拦截请点允许）")

## 训完之后（回本地）

```bash
cd /Users/spark/Documents/Kimi/Workspaces/duckbot
.venv/bin/python trex_sim/run_walk.py \
  --model trex_sim/scene_trex.xml \
  --onnx <下载解压后的策略.onnx> \
  --seconds 10 --vx 0.1 --out trex_walk_trained.mp4
```

验收标准：30 秒行走不摔倒 + 转弯稳定 + 轻推扰动可恢复 → 过阶段 0 门禁，进入硬件采购。